# 🌿 Plant Disease Detector

**Goal:** take a photo of a tomato leaf and identify which disease (if any) it has — then deploy it as a web app you can open on your phone.

**New skills in this project**
- **Transfer learning** — reusing a model (ResNet) that was already trained on 1.2 million images
- **Feature extraction** vs **fine-tuning**
- Working with a real **image dataset** in folders
- **Train / validation / test** splits for deep learning
- **Deploying** a model as a phone-friendly web app (Gradio, Hugging Face Spaces)
- Thinking about **dataset bias** — lab photos vs real fields

⏱️ Two to three weeks. **How to run:** open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file. Run cells top to bottom with **Shift + Enter**.

⚡ **Turn on the free GPU first:** in Colab, **Runtime → Change runtime type → T4 GPU → Save**. Training will be 10–50× faster.

## 1. Setup

In [ ]:
import os, json, random, subprocess
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using", device)
if device == "cpu":
    print("⚠️ No GPU — go to Runtime → Change runtime type → T4 GPU, then run again.")
torch.manual_seed(42); random.seed(42); np.random.seed(42)

## 2. Get the PlantVillage dataset

**PlantVillage** has ~54,000 leaf photos of 14 crops and 26 diseases. To keep things quick we use only the **10 tomato classes** (~18,000 images, 9 diseases + healthy).

We download just the tomato folders from GitHub using a *sparse checkout* (this takes 2–5 minutes).

In [ ]:
DATA_DIR = "PlantVillage-Dataset/raw/color"
if not os.path.exists(DATA_DIR):
    cmds = [
        "git clone --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git",
        "cd PlantVillage-Dataset && git sparse-checkout set --no-cone 'raw/color/Tomato*'",
    ]
    for c in cmds:
        print(">", c)
        subprocess.run(c, shell=True, check=True)
classes = sorted(d for d in os.listdir(DATA_DIR) if d.startswith("Tomato"))
for c in classes:
    print(f"{len(os.listdir(os.path.join(DATA_DIR, c))):5d}  {c}")

*If the download fails:* get the dataset from Kaggle instead — run `import kagglehub; path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")`, then set `DATA_DIR` to the `color` folder inside `path`.

## 3. Look at the images

In [ ]:
def pretty(name):
    plant, _, disease = name.partition("___")
    return f"{plant}: {disease.replace('_', ' ').strip()}"

from PIL import Image
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for ax, c in zip(axes.flat, classes):
    f = sorted(os.listdir(os.path.join(DATA_DIR, c)))[0]
    ax.imshow(Image.open(os.path.join(DATA_DIR, c, f)))
    ax.set_title(pretty(c).replace("Tomato: ", ""), fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

👀 Notice every photo is a **single leaf on a plain background**, taken in a lab. Keep that in mind — we'll come back to it.

## 4. Prepare the images

Pretrained models expect images the same way they saw them during their original training: **224×224 pixels**, normalised with the ImageNet averages.

For training images we add **augmentation** (random crops, flips, rotation, colour changes) so the model learns the *disease*, not the exact photo.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

is_tomato = lambda path: os.path.basename(os.path.dirname(path)).startswith("Tomato")
train_view = datasets.ImageFolder(DATA_DIR, transform=train_tf, is_valid_file=lambda p: is_tomato(p) and p.lower().endswith((".jpg", ".jpeg", ".png")))
eval_view = datasets.ImageFolder(DATA_DIR, transform=eval_tf, is_valid_file=lambda p: is_tomato(p) and p.lower().endswith((".jpg", ".jpeg", ".png")))
CLASS_NAMES = train_view.classes
print(len(train_view), "images,", len(CLASS_NAMES), "classes")

### Train / validation / test split
- **Train (70%)** — the model learns from these.
- **Validation (15%)** — we check progress after each epoch and pick the best version.
- **Test (15%)** — touched **once** at the very end.

`MAX_PER_CLASS` lets you train on a smaller sample first (faster); set it to `None` to use everything.

In [ ]:
MAX_PER_CLASS = 600     # ~6,000 images: a few minutes per epoch on a T4. Use None for all ~18,000.

targets = np.array(train_view.targets)
train_idx, val_idx, test_idx = [], [], []
for c in range(len(CLASS_NAMES)):
    idx = np.where(targets == c)[0]
    np.random.shuffle(idx)
    if MAX_PER_CLASS:
        idx = idx[:MAX_PER_CLASS]
    n_tr, n_va = int(0.7 * len(idx)), int(0.15 * len(idx))
    train_idx += idx[:n_tr].tolist(); val_idx += idx[n_tr:n_tr + n_va].tolist(); test_idx += idx[n_tr + n_va:].tolist()

train_ds = Subset(train_view, train_idx)
val_ds, test_ds = Subset(eval_view, val_idx), Subset(eval_view, test_idx)
BATCH = 64
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=(device == "cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH, num_workers=2)
print(len(train_ds), "train /", len(val_ds), "val /", len(test_ds), "test")

## 5. Transfer learning

Training a big image model from scratch needs millions of images and days of GPU time. Instead we borrow **ResNet-18**, already trained on **ImageNet** (1.2 million photos, 1,000 everyday categories).

Its early layers already detect edges, textures, spots and colours — exactly what disease symptoms are made of. We just **replace the last layer** (which picked among 1,000 ImageNet classes) with a new one for our 10 tomato classes.

```
[pretrained ResNet layers — "eyes"]  →  [NEW final layer: 10 tomato classes]
```

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
print("Original last layer:", model.fc)
model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
print("New last layer:     ", model.fc)
model = model.to(device)

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = nn.functional.cross_entropy(out, y)
            if training:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(y); n += len(y)
            preds = out.argmax(1)
            correct += (preds == y).sum().item()
            all_preds.append(preds.cpu()); all_labels.append(y.cpu())
    return total_loss / n, correct / n, torch.cat(all_preds), torch.cat(all_labels)

def fit(model, optimizer, epochs, best_path="plant_model.pt"):
    best = 0.0
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc, _, _ = run_epoch(model, train_loader, optimizer)
        va_loss, va_acc, _, _ = run_epoch(model, val_loader)
        flag = ""
        if va_acc > best:
            best = va_acc; torch.save(model.state_dict(), best_path); flag = "  ← best so far, saved"
        print(f"epoch {epoch}: train acc {tr_acc:.1%} | val acc {va_acc:.1%}{flag}")
    return best

### Stage 1 — Feature extraction (freeze the "eyes")
We **freeze** all the pretrained layers and train only the new last layer. It's fast and already works well.

In [ ]:
for p in model.parameters():
    p.requires_grad = False
for p in model.fc.parameters():
    p.requires_grad = True
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Training only {trainable:,} of {sum(p.numel() for p in model.parameters()):,} numbers")

_ = fit(model, torch.optim.Adam(model.fc.parameters(), lr=1e-3), epochs=3)

### Stage 2 — Fine-tuning (gently adjust everything)
Now we **unfreeze** all layers and train with a **much smaller learning rate**, so the pretrained knowledge is adjusted for leaves rather than overwritten.

In [ ]:
for p in model.parameters():
    p.requires_grad = True
best_val = fit(model, torch.optim.Adam(model.parameters(), lr=1e-4), epochs=3)
print(f"Best validation accuracy: {best_val:.1%}")

## 6. Final test

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

model.load_state_dict(torch.load("plant_model.pt", map_location=device))
_, test_acc, preds, labels = run_epoch(model, test_loader)
print(f"Test accuracy: {test_acc:.1%}\n")
short_names = [pretty(c).replace("Tomato: ", "")[:22] for c in CLASS_NAMES]
print(classification_report(labels, preds, target_names=short_names, digits=3, zero_division=0))
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay.from_predictions(labels, preds, display_labels=short_names, ax=ax,
                                        cmap="Greens", colorbar=False, xticks_rotation=60)
plt.tight_layout(); plt.show()

In [ ]:
# Show some test predictions
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
inv = transforms.Normalize([-m / s for m, s in zip(MEAN, STD)], [1 / s for s in STD])
for ax, i in zip(axes.flat, np.random.choice(len(test_ds), 12, replace=False)):
    x, y = test_ds[i]
    with torch.no_grad():
        p = model(x.unsqueeze(0).to(device)).argmax(1).item()
    ax.imshow(inv(x).permute(1, 2, 0).clamp(0, 1))
    ok = p == y
    ax.set_title(("✓ " if ok else "✗ ") + short_names[p], color="green" if ok else "red", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 7. ⚠️ A reality check: dataset bias

~99% accuracy looks amazing — but remember the photos: **one leaf, plain background, good lighting**. Research has shown PlantVillage models can drop to **30–60% accuracy on real field photos** (messy backgrounds, several leaves, shadows).

The model may also have learned shortcuts — e.g. background colour or camera settings that happened to differ between classes. **High test accuracy only means "works on data like the test set".**

How to make it robust in real life:
- Test on your own photos (next section).
- Train with real-world images too (see [PlantDoc](https://github.com/pratikkayal/PlantDoc-Dataset)).
- Use stronger augmentation (random backgrounds, blur, lighting).

## 8. Try your own leaf photo
Upload a photo in Colab's 📁 Files panel (left side) and put its filename below.

In [ ]:
def diagnose(img):
    """img: a PIL image or numpy array → dict of {disease: probability}"""
    if img is None:
        return {}
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)
    x = eval_tf(img.convert("RGB")).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0].cpu()
    return {pretty(c): float(p) for c, p in zip(CLASS_NAMES, probs)}

MY_PHOTO = "my_leaf.jpg"          # ✏️ change to your file name
if os.path.exists(MY_PHOTO):
    res = sorted(diagnose(Image.open(MY_PHOTO)).items(), key=lambda kv: -kv[1])[:3]
    for name, p in res:
        print(f"{p:6.1%}  {name}")
else:
    # Demo with a test image instead
    x_path = eval_view.samples[test_idx[0]][0]
    print("Demo on", os.path.basename(os.path.dirname(x_path)))
    for name, p in sorted(diagnose(Image.open(x_path)).items(), key=lambda kv: -kv[1])[:3]:
        print(f"{p:6.1%}  {name}")

## 9. 📱 Deploy as a web app you can open on your phone

**Quick (72-hour link):** run the cell below and open the `gradio.live` link on your phone. You can take a photo with your phone camera and upload it.

In [ ]:
import gradio as gr

app = gr.Interface(
    fn=diagnose,
    inputs=gr.Image(type="numpy", sources=["upload", "webcam"], label="Tomato leaf photo"),
    outputs=gr.Label(num_top_classes=3, label="Diagnosis"),
    title="🍅 Tomato Leaf Disease Detector",
    description="Upload or take a photo of a single tomato leaf.",
)
# app.launch(share=True)   # ← remove the # to launch and get a public link

**Permanent (free) deployment on Hugging Face Spaces:**
1. Save the files below (next cell) and download `plant_model.pt` and `class_names.json` from the Files panel.
2. Create a free account at [huggingface.co](https://huggingface.co) → **New Space** → SDK: **Gradio** → hardware: free CPU.
3. Upload `app.py`, `requirements.txt` (both in this project folder), `plant_model.pt` and `class_names.json`.
4. Wait a minute for it to build — you get a permanent link like `huggingface.co/spaces/<you>/tomato-doctor` that works on any phone.

In [ ]:
with open("class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f)
print("Saved class_names.json and plant_model.pt — download both for deployment")

## ✅ What you learned
- **Transfer learning**: reuse a pretrained network, replace the last layer.
- **Feature extraction** (frozen backbone) then **fine-tuning** (small learning rate).
- **Train / val / test** discipline and saving the **best** checkpoint.
- **Dataset bias**: great test scores ≠ works in the real world.
- **Deployment** with Gradio and Hugging Face Spaces.

## ✍️ Try it yourself
1. Swap ResNet-18 for `models.efficientnet_b0` or `models.mobilenet_v3_small` (mobile-friendly). Hint: their last layer is `model.classifier[-1]`.
2. Train on **all 38 PlantVillage classes** (change the sparse-checkout pattern to `'raw/color/*'` — about 1 GB).
3. Add **Grad-CAM** (`pip install grad-cam`) to highlight *where* on the leaf the model is looking. Is it looking at the spots — or the background?
4. Photograph 10 real leaves from a garden or market and measure the accuracy honestly.

In [ ]:
# Your code here